# Dataset Verification and Overview

**Project:** Context-Aware Trust Scoring and Review-Based Product Recommendation  
**Dataset:** Amazon Reviews 2018 (Fashion Category)

**Objective:**  
To verify that the dataset is usable, well-structured, and suitable for trust scoring and recommendation tasks.


Import Required Libraries

In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_colwidth', 300)
pd.set_option('display.max_columns', None)


Load Dataset

In [3]:
import os
import gzip
import json
import shutil
import requests
from pathlib import Path

# Define paths relative to the project root
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PATH = DATA_DIR / "AMAZON_FASHION.json"
DATA_URL = "http://deepyeti.ucsd.edu/jianmo/amazon/categoryFiles/AMAZON_FASHION.json.gz"

# Ensure directory exists
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Download and unzip if file doesn't exist
if not DATA_PATH.exists():
    print(f"File not found at {DATA_PATH}. Downloading from {DATA_URL}...")
    try:
        response = requests.get(DATA_URL, stream=True)
        response.raise_for_status()
        
        # Download compressed file
        compressed_path = DATA_PATH.with_suffix(".json.gz")
        with open(compressed_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
        print("Download complete. Extracting...")
        
        # Extract json.gz to json
        with gzip.open(compressed_path, 'rb') as f_in:
            with open(DATA_PATH, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        
        # Clean up compressed file
        compressed_path.unlink()
        print("Extraction complete.")
        
    except Exception as e:
        print(f"Error downloading/extracting dataset: {e}")
        raise

print(f"Loading dataset from: {DATA_PATH}")
df = pd.read_json(DATA_PATH, lines=True)
print("Dataset loaded successfully!")


Loading dataset from: D:\Context_Aware_Trust_Scoring_Recommendation_Fashion\data\raw\AMAZON_FASHION.json
Dataset loaded successfully!


In [4]:
print(f"Total number of reviews: {df.shape[0]}")


Total number of reviews: 883636


In [5]:
print("Column names:\n")
print(df.columns.tolist())


Column names:

['overall', 'verified', 'reviewTime', 'reviewerID', 'asin', 'reviewerName', 'reviewText', 'summary', 'unixReviewTime', 'vote', 'style', 'image']


In [6]:
df.dtypes


overall             int64
verified             bool
reviewTime         object
reviewerID         object
asin               object
reviewerName       object
reviewText         object
summary            object
unixReviewTime      int64
vote              float64
style              object
image              object
dtype: object

In [7]:
df.sample(5)[
    ["reviewerID", "asin", "overall", "reviewText", "summary", "verified"]
]


,reviewerID,asin,overall,reviewText,summary,verified
564951,ADINNKTTBXBUL,B00O9X2KH0,5,"It's so soft and really hot , which is what I wanted because I stay cold. It's super soft inside and out and the drop crotch isn't too low at all. Nothing has fallen off and I can't spot any loose threads or seams. The feet in the picture don't come with it! I suggest ordering a size up from you...",IM IN LOVE!!!!!,True
561150,A1BW41C6UGN65M,B00NW84QW0,5,Love it great to wear to work or the bar!,Five Stars,True
160556,A26IH53Z5PZ8O9,B00HH970VY,5,The quality is so much better than I would have expected for a bracelet that cost so little. It's beautiful.,Better than pictured,True
187021,AGI1CT3U7KLR2,B00K2LP0S4,1,You would have be under 5' for it to be a maxi dress. This dress was nothing like the picture. The slip was sewn in wrong. Very disappointed. Will not buy from this seller again.,OMG,True
225468,A1TW9MNGQCJ66R,B00OVJU9ES,1,"Junk bought purple , arrived purple two day after taking out of pkg they turned clear",Junk,True


In [8]:
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_percentage.sort_values(ascending=False)


image             96.739947
vote              90.957815
style             65.532301
reviewText         0.139537
summary            0.060319
reviewerName       0.010412
verified           0.000000
overall            0.000000
reviewerID         0.000000
reviewTime         0.000000
asin               0.000000
unixReviewTime     0.000000
dtype: float64

In [9]:
# Reviews per user
reviews_per_user = df["reviewerID"].value_counts()
print("Users with only 1 review:", (reviews_per_user == 1).sum())

# Reviews per product
reviews_per_product = df["asin"].value_counts()
print("Products with only 1 review:", (reviews_per_product == 1).sum())

Users with only 1 review: 655320
Products with only 1 review: 99962


## Dataset Summary

- The Amazon Fashion dataset contains user reviews with textual feedback and ratings.
- Each review is associated with a unique user (`reviewerID`) and product (`asin`).
- Ratings are numeric, and reviews include rich free-text content.
- Additional metadata such as verification status and timestamps are available.
- The dataset is suitable for:
  - Review-level trust scoring
  - Product-level recommendation aggregation
